# 85. The Uncapacitated Facility Location Problem
## Tier 2: The Classic Heuristic (Cross Entropy Method)

### Key assumptions
- Facilities have no capacity constraints
- Each customer must be assigned to exactly one facility
- Transportation costs are linear and deterministic
- Cross Entropy Method can find near-optimal solutions through probabilistic sampling
- Elite solutions guide the probability distribution evolution

### Approach (step-by-step)
1. **Initialize probability distribution** over facility openings
   - Start with uniform probability for each facility
   - Each facility has independent probability of being opened

2. **Generate sample solutions** probabilistically
   - Sample facility configurations according to current probabilities
   - Assign customers to cheapest open facilities
   - Calculate total costs for each solution

3. **Select elite solutions** and update distribution
   - Keep top ρ percentile of solutions as elite set
   - Update probabilities based on elite solution statistics
   - Apply smoothing parameter α for gradual learning

4. **Iterate until convergence** or maximum iterations
   - Monitor convergence through best solution improvement
   - Track probability evolution and solution quality

### What to look for in the results
- Probability evolution showing convergence to optimal facility set
- Elite solution quality improvement over iterations
- Final facility opening decisions with confidence levels
- Comparison with mathematical optimum from Tier 1

### Concrete example (from the source)
5-facility, 8-customer instance with:
- Fixed costs: [80, 120, 95, 110, 85]
- Random transportation cost matrix
- Cross Entropy parameters: N=500, ρ=0.2, α=0.8

In [1]:
# Import required libraries for Cross Entropy Method
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(85)

print("Libraries imported successfully for Cross Entropy Method")

Libraries imported successfully for Cross Entropy Method


Libraries imported successfully for Cross Entropy Method


Libraries imported successfully for Cross Entropy Method


Libraries imported successfully for Cross Entropy Method


In [ ]:
@dataclass
class UFLPInstance:
    """Data class for Uncapacitated Facility Location Problem instance"""
    
    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    transport_costs: np.ndarray
    
    def __post_init__(self):
        """Validate data consistency"""
        assert len(self.fixed_costs) == self.n_facilities
        assert self.transport_costs.shape == (self.n_customers, self.n_facilities)

@dataclass
class UFLPSolution:
    """Data class for UFLP solution"""
    
    facilities_open: List[int]  # Binary list of facility opening decisions
    assignments: List[int]      # Facility index for each customer
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool

@dataclass
class CEParameters:
    """Parameters for Cross Entropy Method"""
    
    N: int = 500          # Sample size per iteration
    rho: float = 0.2      # Elite fraction (keep top 20%)
    alpha: float = 0.8    # Smoothing parameter
    max_iter: int = 50    # Maximum iterations
    tol: float = 1e-6     # Convergence tolerance

print("Data structures defined for Cross Entropy Method")

In [ ]:
def create_ce_example_instance():
    """Create the 5-facility, 8-customer instance from the source
    
    Returns:
        UFLPInstance: Problem instance for Cross Entropy demonstration
    """
    # Problem parameters from the concrete example
    n_facilities = 5
    n_customers = 8
    
    # Fixed costs for facilities [80, 120, 95, 110, 85]
    fixed_costs = [80, 120, 95, 110, 85]
    
    # Transportation costs (customers x facilities)
    # Random but reasonable values for demonstration
    np.random.seed(42)  # For reproducible example
    transport_costs = np.random.randint(10, 50, (n_customers, n_facilities))
    
    # Make some routes more expensive to create interesting trade-offs
    transport_costs[0] = [12, 35, 28, 18, 32]
    transport_costs[1] = [25, 15, 22, 30, 18]
    transport_costs[2] = [18, 28, 14, 25, 20]
    transport_costs[3] = [30, 12, 25, 15, 28]
    transport_costs[4] = [22, 18, 30, 12, 25]
    transport_costs[5] = [15, 25, 18, 28, 22]
    transport_costs[6] = [28, 22, 15, 25, 18]
    transport_costs[7] = [20, 30, 22, 18, 15]
    
    return UFLPInstance(
        n_facilities=n_facilities,
        n_customers=n_customers,
        fixed_costs=fixed_costs,
        transport_costs=transport_costs
    )

# Create the Cross Entropy example instance
ce_instance = create_ce_example_instance()

print(f"Created Cross Entropy UFLP instance:")
print(f"  Facilities: {ce_instance.n_facilities}")
print(f"  Customers: {ce_instance.n_customers}")
print(f"  Fixed costs: {ce_instance.fixed_costs}")
print(f"  Transport costs shape: {ce_instance.transport_costs.shape}")

# Display transport cost matrix
print("\nTransport Cost Matrix:")
print("Customer\t", end="")
for j in range(ce_instance.n_facilities):
    print(f"F{j+1}\t", end="")
print()
for i in range(ce_instance.n_customers):
    print(f"C{i+1}\t", end="")
    for j in range(ce_instance.n_facilities):
        print(f"{ce_instance.transport_costs[i,j]}\t", end="")
    print()

In [ ]:
class CrossEntropyMethod:
    """Cross Entropy Method for Uncapacitated Facility Location Problem"""
    
    def __init__(self, instance: UFLPInstance, params: CEParameters):
        self.instance = instance
        self.params = params
        self.probabilities = np.ones(instance.n_facilities) * 0.5  # Start with uniform
        self.best_solution = None
        self.history = {
            'best_costs': [],
            'avg_costs': [],
            'probabilities': [],
            'elite_costs': []
        }
        
    def generate_sample_solution(self) -> UFLPSolution:
        """Generate a single solution by sampling from current probabilities
        
        Returns:
            UFLPSolution: Sampled solution
        """
        # Sample facility opening decisions
        facilities_open = [np.random.random() < self.probabilities[j] 
                          for j in range(self.instance.n_facilities)]
        
        # Ensure at least one facility is open
        if not any(facilities_open):
            # Open facility with highest probability
            max_prob_idx = np.argmax(self.probabilities)
            facilities_open[max_prob_idx] = True
        
        # Assign customers to cheapest open facilities
        assignments = []
        for i in range(self.instance.n_customers):
            min_cost = float('inf')
            best_facility = -1
            
            for j in range(self.instance.n_facilities):
                if facilities_open[j]:
                    cost = self.instance.transport_costs[i, j]
                    if cost < min_cost:
                        min_cost = cost
                        best_facility = j
            
            assignments.append(best_facility)
        
        # Calculate total cost
        fixed_cost = sum(self.instance.fixed_costs[j] 
                        for j in range(self.instance.n_facilities) if facilities_open[j])
        
        transport_cost = sum(self.instance.transport_costs[i, assignments[i]] 
                           for i in range(self.instance.n_customers))
        
        total_cost = fixed_cost + transport_cost
        
        return UFLPSolution(
            facilities_open=facilities_open,
            assignments=assignments,
            total_cost=total_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=True
        )
    
    def generate_samples(self, N: int) -> List[UFLPSolution]:
        """Generate N sample solutions
        
        Args:
            N: Number of samples to generate
            
        Returns:
            List of N solutions
        """
        samples = []
        for _ in range(N):
            samples.append(self.generate_sample_solution())
        return samples
    
    def select_elite(self, samples: List[UFLPSolution]) -> List[UFLPSolution]:
        """Select elite solutions (top ρ percentile)
        
        Args:
            samples: List of sample solutions
            
        Returns:
            List of elite solutions
        """
        # Sort by cost (ascending - better solutions have lower cost)
        sorted_samples = sorted(samples, key=lambda x: x.total_cost)
        
        # Select top ρ percentile
        n_elite = max(1, int(len(sorted_samples) * self.params.rho))
        elite = sorted_samples[:n_elite]
        
        return elite
    
    def update_probabilities(self, elite: List[UFLPSolution]) -> np.ndarray:
        """Update probability distribution based on elite solutions
        
        Args:
            elite: List of elite solutions
            
        Returns:
            Updated probability vector
        """
        # Calculate empirical probabilities from elite solutions
        new_probabilities = np.zeros(self.instance.n_facilities)
        
        for solution in elite:
            for j in range(self.instance.n_facilities):
                if solution.facilities_open[j]:
                    new_probabilities[j] += 1
        
        new_probabilities /= len(elite)
        
        # Apply smoothing: p_new = α * p_empirical + (1-α) * p_old
        smoothed_probabilities = (
            self.params.alpha * new_probabilities + 
            (1 - self.params.alpha) * self.probabilities
        )
        
        # Ensure probabilities stay in [0.01, 0.99] to avoid degeneracy
        smoothed_probabilities = np.clip(smoothed_probabilities, 0.01, 0.99)
        
        return smoothed_probabilities
    
    def check_convergence(self, old_probabilities: np.ndarray, 
                         new_probabilities: np.ndarray) -> bool:
        """Check if probabilities have converged
        
        Args:
            old_probabilities: Previous probability vector
            new_probabilities: Current probability vector
            
        Returns:
            True if converged, False otherwise
        """
        max_change = np.max(np.abs(new_probabilities - old_probabilities))
        return max_change < self.params.tol
    
    def solve(self) -> UFLPSolution:
        """Solve UFLP using Cross Entropy Method
        
        Returns:
            Best solution found
        """
        print("Starting Cross Entropy Method for UFLP...")
        print(f"Parameters: N={self.params.N}, ρ={self.params.rho}, α={self.params.alpha}")
        
        for iteration in range(self.params.max_iter):
            # Generate samples
            samples = self.generate_samples(self.params.N)
            
            # Track statistics
            costs = [s.total_cost for s in samples]
            avg_cost = np.mean(costs)
            best_cost = min(costs)
            
            # Select elite solutions
            elite = self.select_elite(samples)
            elite_costs = [s.total_cost for s in elite]
            elite_avg = np.mean(elite_costs)
            
            # Update best solution
            current_best = elite[0]  # Elite is sorted by cost
            if self.best_solution is None or current_best.total_cost < self.best_solution.total_cost:
                self.best_solution = current_best
            
            # Store history
            self.history['best_costs'].append(best_cost)
            self.history['avg_costs'].append(avg_cost)
            self.history['probabilities'].append(self.probabilities.copy())
            self.history['elite_costs'].append(elite_avg)
            
            # Print progress
            if (iteration + 1) % 5 == 0 or iteration == 0:
                print(f"Iteration {iteration + 1}: Best = {best_cost:.1f}, "
                      f"Elite Avg = {elite_avg:.1f}, "
                      f"Overall Avg = {avg_cost:.1f}")
            
            # Update probabilities
            old_probabilities = self.probabilities.copy()
            self.probabilities = self.update_probabilities(elite)
            
            # Check convergence
            if self.check_convergence(old_probabilities, self.probabilities):
                print(f"Converged at iteration {iteration + 1}")
                break
        
        print(f"Completed {iteration + 1} iterations")
        return self.best_solution

print("Cross Entropy Method class defined")

In [ ]:
# Solve the UFLP using Cross Entropy Method
ce_params = CEParameters(N=500, rho=0.2, alpha=0.8, max_iter=50)
ce_solver = CrossEntropyMethod(ce_instance, ce_params)
ce_solution = ce_solver.solve()

print("\n=== CROSS ENTROPY SOLUTION RESULTS ===")
print(f"Total cost: {ce_solution.total_cost:.2f}")
print(f"Fixed cost: {ce_solution.fixed_cost:.2f}")
print(f"Transport cost: {ce_solution.transport_cost:.2f}")
print(f"\nFacilities opened: {[j+1 for j, open in enumerate(ce_solution.facilities_open) if open]}")
print(f"Number of facilities open: {sum(ce_solution.facilities_open)}")

print("\nCustomer assignments:")
for i, facility in enumerate(ce_solution.assignments):
    cost = ce_instance.transport_costs[i, facility]
    print(f"  Customer {i+1} -> Facility {facility+1} (Cost: {cost})")

print(f"\nFinal probabilities: {ce_solver.probabilities}")

In [ ]:
def visualize_ce_results(ce_solver: CrossEntropyMethod, solution: UFLPSolution):
    """Create comprehensive visualization of Cross Entropy Method results
    
    Args:
        ce_solver: Cross Entropy solver with history
        solution: Best solution found
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Cross Entropy Method - UFLP Solution Analysis', fontsize=16, fontweight='bold')
    
    # 1. Convergence plot
    ax1 = axes[0, 0]
    iterations = range(1, len(ce_solver.history['best_costs']) + 1)
    ax1.plot(iterations, ce_solver.history['best_costs'], 'o-', linewidth=2, 
            markersize=6, color='#2E86AB', label='Best Cost')
    ax1.plot(iterations, ce_solver.history['elite_costs'], 's--', linewidth=2, 
            markersize=6, color='#A23B72', label='Elite Average')
    ax1.plot(iterations, ce_solver.history['avg_costs'], 'd:', linewidth=2, 
            markersize=6, color='#F18F01', label='Overall Average')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Total Cost')
    ax1.set_title('Cost Convergence')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Probability evolution
    ax2 = axes[0, 1]
    prob_matrix = np.array(ce_solver.history['probabilities']).T
    facility_labels = [f'F{j+1}' for j in range(ce_solver.instance.n_facilities)]
    
    for j in range(ce_solver.instance.n_facilities):
        ax2.plot(iterations, prob_matrix[j], 'o-', linewidth=2, 
                markersize=4, label=facility_labels[j])
    
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Opening Probability')
    ax2.set_title('Probability Evolution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)
    
    # 3. Final facility decisions
    ax3 = axes[1, 0]
    facility_numbers = [f'F{j+1}' for j in range(ce_solver.instance.n_facilities)]
    opening_status = ['Open' if open_fac else 'Closed' for open_fac in solution.facilities_open]
    colors_fac = ['#F18F01' if open_fac else '#C73E1D' for open_fac in solution.facilities_open]
    
    bars = ax3.bar(facility_numbers, [1 if open_fac else 0] * ce_solver.instance.n_facilities, color=colors_fac)
    ax3.set_title('Final Facility Opening Decisions')
    ax3.set_ylabel('Open (1) / Closed (0)')
    ax3.set_ylim(-0.1, 1.1)
    
    # Add fixed costs and final probabilities as text
    for i, (bar, open_fac, cost, prob) in enumerate(zip(bars, solution.facilities_open, 
                                                       ce_solver.instance.fixed_costs, 
                                                       ce_solver.probabilities)):
        ax3.text(bar.get_x() + bar.get_width()/2, 0.5, f'${cost}k\n({prob:.2f})', 
                ha='center', va='center', fontweight='bold', fontsize=8)
    
    # 4. Cost breakdown
    ax4 = axes[1, 1]
    costs = [solution.fixed_cost, solution.transport_cost]
    labels = ['Fixed Cost', 'Transport Cost']
    colors = ['#2E86AB', '#A23B72']
    
    wedges, texts, autotexts = ax4.pie(costs, labels=labels, colors=colors, 
                                      autopct='%1.1f%%', startangle=90)
    ax4.set_title(f'Cost Breakdown\n(Total: ${solution.total_cost:.0f}k)')
    
    plt.tight_layout()
    plt.show()
    
    # Additional analysis: Probability confidence intervals
    print("\n=== PROBABILITY ANALYSIS ===")
    print("Final facility opening probabilities:")
    for j in range(ce_solver.instance.n_facilities):
        prob = ce_solver.probabilities[j]
        status = "OPEN" if solution.facilities_open[j] else "CLOSED"
        confidence = "HIGH" if prob > 0.8 or prob < 0.2 else "MEDIUM" if prob > 0.6 or prob < 0.4 else "LOW"
        print(f"  Facility {j+1}: {prob:.3f} -> {status} (Confidence: {confidence})")

# Visualize the Cross Entropy results
visualize_ce_results(ce_solver, ce_solution)

In [ ]:
def parameter_sensitivity_analysis(instance: UFLPInstance):
    """Analyze sensitivity of Cross Entropy Method to parameters
    
    Args:
        instance: UFLP instance to analyze
    """
    print("=== PARAMETER SENSITIVITY ANALYSIS ===")
    
    # Test different sample sizes N
    print("\n1. Sample Size (N) Sensitivity:")
    sample_sizes = [100, 300, 500, 700, 1000]
    n_results = []
    
    for N in sample_sizes:
        params = CEParameters(N=N, rho=0.2, alpha=0.8, max_iter=30)
        solver = CrossEntropyMethod(instance, params)
        solution = solver.solve()
        
        n_results.append({
            'N': N,
            'cost': solution.total_cost,
            'facilities': sum(solution.facilities_open),
            'iterations': len(solver.history['best_costs'])
        })
        
        print(f"  N={N}: Cost={solution.total_cost:.1f}, Facilities={sum(solution.facilities_open)}, Iterations={len(solver.history['best_costs'])}")
    
    # Test different elite fractions rho
    print("\n2. Elite Fraction (ρ) Sensitivity:")
    rho_values = [0.1, 0.15, 0.2, 0.25, 0.3]
    rho_results = []
    
    for rho in rho_values:
        params = CEParameters(N=500, rho=rho, alpha=0.8, max_iter=30)
        solver = CrossEntropyMethod(instance, params)
        solution = solver.solve()
        
        rho_results.append({
            'rho': rho,
            'cost': solution.total_cost,
            'facilities': sum(solution.facilities_open),
            'iterations': len(solver.history['best_costs'])
        })
        
        print(f"  ρ={rho:.2f}: Cost={solution.total_cost:.1f}, Facilities={sum(solution.facilities_open)}, Iterations={len(solver.history['best_costs'])}")
    
    # Test different smoothing parameters alpha
    print("\n3. Smoothing Parameter (α) Sensitivity:")
    alpha_values = [0.5, 0.6, 0.7, 0.8, 0.9]
    alpha_results = []
    
    for alpha in alpha_values:
        params = CEParameters(N=500, rho=0.2, alpha=alpha, max_iter=30)
        solver = CrossEntropyMethod(instance, params)
        solution = solver.solve()
        
        alpha_results.append({
            'alpha': alpha,
            'cost': solution.total_cost,
            'facilities': sum(solution.facilities_open),
            'iterations': len(solver.history['best_costs'])
        })
        
        print(f"  α={alpha:.1f}: Cost={solution.total_cost:.1f}, Facilities={sum(solution.facilities_open)}, Iterations={len(solver.history['best_costs'])}")
    
    # Create sensitivity plots
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    # Sample size sensitivity
    N_values = [r['N'] for r in n_results]
    costs_N = [r['cost'] for r in n_results]
    facilities_N = [r['facilities'] for r in n_results]
    
    ax1.plot(N_values, costs_N, 'o-', linewidth=2, markersize=8, color='#2E86AB')
    ax1.set_xlabel('Sample Size (N)')
    ax1.set_ylabel('Total Cost', color='#2E86AB')
    ax1.tick_params(axis='y', labelcolor='#2E86AB')
    ax1.set_title('Sample Size Sensitivity')
    ax1.grid(True, alpha=0.3)
    
    ax1_twin = ax1.twinx()
    ax1_twin.plot(N_values, facilities_N, 's--', linewidth=2, markersize=8, color='#A23B72')
    ax1_twin.set_ylabel('Number of Facilities', color='#A23B72')
    ax1_twin.tick_params(axis='y', labelcolor='#A23B72')
    
    # Elite fraction sensitivity
    rho_vals = [r['rho'] for r in rho_results]
    costs_rho = [r['cost'] for r in rho_results]
    facilities_rho = [r['facilities'] for r in rho_results]
    
    ax2.plot(rho_vals, costs_rho, 'o-', linewidth=2, markersize=8, color='#2E86AB')
    ax2.set_xlabel('Elite Fraction (ρ)')
    ax2.set_ylabel('Total Cost', color='#2E86AB')
    ax2.tick_params(axis='y', labelcolor='#2E86AB')
    ax2.set_title('Elite Fraction Sensitivity')
    ax2.grid(True, alpha=0.3)
    
    ax2_twin = ax2.twinx()
    ax2_twin.plot(rho_vals, facilities_rho, 's--', linewidth=2, markersize=8, color='#A23B72')
    ax2_twin.set_ylabel('Number of Facilities', color='#A23B72')
    ax2_twin.tick_params(axis='y', labelcolor='#A23B72')
    
    # Smoothing parameter sensitivity
    alpha_vals = [r['alpha'] for r in alpha_results]
    costs_alpha = [r['cost'] for r in alpha_results]
    facilities_alpha = [r['facilities'] for r in alpha_results]
    
    ax3.plot(alpha_vals, costs_alpha, 'o-', linewidth=2, markersize=8, color='#2E86AB')
    ax3.set_xlabel('Smoothing Parameter (α)')
    ax3.set_ylabel('Total Cost', color='#2E86AB')
    ax3.tick_params(axis='y', labelcolor='#2E86AB')
    ax3.set_title('Smoothing Parameter Sensitivity')
    ax3.grid(True, alpha=0.3)
    
    ax3_twin = ax3.twinx()
    ax3_twin.plot(alpha_vals, facilities_alpha, 's--', linewidth=2, markersize=8, color='#A23B72')
    ax3_twin.set_ylabel('Number of Facilities', color='#A23B72')
    ax3_twin.tick_params(axis='y', labelcolor='#A23B72')
    
    plt.tight_layout()
    plt.show()

# Perform parameter sensitivity analysis
parameter_sensitivity_analysis(ce_instance)

In [ ]:
def compare_ce_with_baseline(instance: UFLPInstance, ce_solution: UFLPSolution):
    """Compare Cross Entropy solution with simple heuristics
    
    Args:
        instance: UFLP instance
        ce_solution: Cross Entropy solution
    """
    def greedy_heuristic():
        """Simple greedy heuristic: open facilities with minimum average cost"""
        # Calculate average transport cost per facility
        avg_transport_costs = []
        for j in range(instance.n_facilities):
            avg_cost = np.mean(instance.transport_costs[:, j])
            total_avg_cost = instance.fixed_costs[j] + avg_cost * instance.n_customers
            avg_transport_costs.append(total_avg_cost)
        
        # Sort facilities by average cost
        facility_order = np.argsort(avg_transport_costs)
        
        # Greedily open facilities
        facilities_open = [False] * instance.n_facilities
        assignments = [-1] * instance.n_customers
        
        for j in facility_order:
            # Check if opening this facility improves any assignment
            improved = False
            for i in range(instance.n_customers):
                current_cost = (instance.transport_costs[i, assignments[i]] 
                               if assignments[i] >= 0 else float('inf'))
                new_cost = instance.transport_costs[i, j]
                if new_cost < current_cost:
                    assignments[i] = j
                    improved = True
            
            if improved:
                facilities_open[j] = True
        
        # Calculate total cost
        fixed_cost = sum(instance.fixed_costs[j] 
                        for j in range(instance.n_facilities) if facilities_open[j])
        transport_cost = sum(instance.transport_costs[i, assignments[i]] 
                           for i in range(instance.n_customers))
        total_cost = fixed_cost + transport_cost
        
        return UFLPSolution(
            facilities_open=facilities_open,
            assignments=assignments,
            total_cost=total_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=True
        )
    
    def random_heuristic():
        """Random heuristic for baseline comparison"""
        best_solution = None
        best_cost = float('inf')
        
        for _ in range(1000):  # Try 1000 random configurations
            # Random facility opening
            facilities_open = [np.random.random() < 0.5 for _ in range(instance.n_facilities)]
            
            # Ensure at least one facility open
            if not any(facilities_open):
                facilities_open[np.random.randint(instance.n_facilities)] = True
            
            # Assign customers to cheapest open facilities
            assignments = []
            for i in range(instance.n_customers):
                min_cost = float('inf')
                best_facility = -1
                for j in range(instance.n_facilities):
                    if facilities_open[j]:
                        cost = instance.transport_costs[i, j]
                        if cost < min_cost:
                            min_cost = cost
                            best_facility = j
                assignments.append(best_facility)
            
            # Calculate cost
            fixed_cost = sum(instance.fixed_costs[j] 
                            for j in range(instance.n_facilities) if facilities_open[j])
            transport_cost = sum(instance.transport_costs[i, assignments[i]] 
                               for i in range(instance.n_customers))
            total_cost = fixed_cost + transport_cost
            
            if total_cost < best_cost:
                best_cost = total_cost
                best_solution = UFLPSolution(
                    facilities_open=facilities_open,
                    assignments=assignments,
                    total_cost=total_cost,
                    fixed_cost=fixed_cost,
                    transport_cost=transport_cost,
                    feasible=True
                )
        
        return best_solution
    
    # Get baseline solutions
    greedy_solution = greedy_heuristic()
    random_solution = random_heuristic()
    
    print("=== CROSS ENTROPY VS BASELINE COMPARISON ===")
    
    solutions = [
        ("Cross Entropy", ce_solution),
        ("Greedy", greedy_solution),
        ("Random Best", random_solution)
    ]
    
    print(f"\n{'Method':<15} {'Total Cost':<12} {'Fixed':<10} {'Transport':<12} {'Facilities':<12}")
    print("-" * 65)
    
    for name, sol in solutions:
        print(f"{name:<15} {sol.total_cost:<12.1f} {sol.fixed_cost:<10.1f} "
              f"{sol.transport_cost:<12.1f} {sum(sol.facilities_open):<12}")
    
    # Calculate improvement percentages
    ce_cost = ce_solution.total_cost
    greedy_cost = greedy_solution.total_cost
    random_cost = random_solution.total_cost
    
    greedy_improvement = ((greedy_cost - ce_cost) / greedy_cost) * 100
    random_improvement = ((random_cost - ce_cost) / random_cost) * 100
    
    print(f"\nImprovement over Greedy: {greedy_improvement:.2f}%")
    print(f"Improvement over Random: {random_improvement:.2f}%")
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Cost comparison
    methods = [name for name, _ in solutions]
    total_costs = [sol.total_cost for _, sol in solutions]
    fixed_costs = [sol.fixed_cost for _, sol in solutions]
    transport_costs = [sol.transport_cost for _, sol in solutions]
    
    x = np.arange(len(methods))
    width = 0.25
    
    ax1.bar(x - width, total_costs, width, label='Total Cost', color='#2E86AB')
    ax1.bar(x, fixed_costs, width, label='Fixed Cost', color='#A23B72')
    ax1.bar(x + width, transport_costs, width, label='Transport Cost', color='#F18F01')
    
    ax1.set_xlabel('Solution Method')
    ax1.set_ylabel('Cost')
    ax1.set_title('Cost Comparison: Cross Entropy vs Baselines')
    ax1.set_xticks(x)
    ax1.set_xticklabels(methods)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Facility count comparison
    facility_counts = [sum(sol.facilities_open) for _, sol in solutions]
    colors_fac = ['#2E86AB', '#A23B72', '#F18F01']
    
    ax2.bar(methods, facility_counts, color=colors_fac)
    ax2.set_xlabel('Solution Method')
    ax2.set_ylabel('Number of Facilities Open')
    ax2.set_title('Facility Count Comparison')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Compare Cross Entropy with baseline methods
compare_ce_with_baseline(ce_instance, ce_solution)

### Why this Tier exists vs earlier Tiers
This Tier 2 introduces the **Cross Entropy Method**, a sophisticated probabilistic heuristic that bridges the gap between exact mathematical optimization and pure heuristics. Unlike the CSP formulation in Tier 1 which guarantees optimality but has exponential complexity, the Cross Entropy Method provides near-optimal solutions with polynomial time complexity.

**Advantages vs Tier 1 (CSP):**
- **Polynomial complexity**: O(T × N × |C| × |F|) vs exponential for CSP
- **Scalability**: Handles much larger instances efficiently
- **Probabilistic robustness**: Less sensitive to problem structure
- **Parallelizable**: Sample generation can be parallelized

**Advantages vs simple heuristics:**
- **Systematic learning**: Adapts probability distribution based on elite solutions
- **Global search**: Avoids getting trapped in poor local optima
- **Theoretical foundation**: Based on principles of cross entropy minimization
- **Convergence guarantees**: Theoretical convergence to optimal distribution

**Disadvantages vs Tier 1:**
- **No optimality guarantee**: May not find the true optimum
- **Parameter sensitivity**: Performance depends on parameter tuning
- **Stochastic results**: Different runs may produce different solutions

**When to use this Tier:**
- Medium to large instances where exact methods are infeasible
- Problems requiring good solutions quickly
- Situations where some optimality gap is acceptable
- Dynamic problems requiring repeated solving

### Pros / Cons Summary
**Pros:**
✓ Excellent scalability to large problem instances
✓ Systematic learning from elite solutions
✓ Strong theoretical foundation
✓ Parallelizable implementation
✓ Robust to problem variations
✓ Provides probability-based insights

**Cons:**
✗ No optimality guarantee
✗ Requires parameter tuning
✗ Stochastic results (non-deterministic)
✗ May converge slowly for some problems
✗ Memory requirements for storing samples

The Cross Entropy Method provides an excellent balance between solution quality and computational efficiency, making it suitable for practical UFLP applications where exact optimality is less critical than scalability and speed.